<a href="https://colab.research.google.com/github/zumrakcck/Fake-News-Detection-with-Deep-Learning/blob/main/FakeNewDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning-Based Fake News Detection & Source-Cue Bias Analysis
## Project Overview & Setup

In this notebook, we implement a baseline BiLSTM model for fake news detection using the ISOT Fake News Dataset.
Our primary objective is twofold:
1. Establish a strong baseline classifier using text embeddings.
2. Investigate the impact of source-cue bias (e.g., agency names like 'Reuters') on model performance by comparing raw texts against a cleaned version where explicit publication sources are removed.


In [7]:
# Import necessary libraries
import os
import re
import shutil
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Check device availability (GPU vs CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running on device: {device}")

# Download ISOT dataset via Kagglehub
import kagglehub
path = kagglehub.dataset_download("emineyetm/fake-news-detection-datasets")
print("Dataset downloaded to:", path)

# Copy CSV files to the local working directory
for file in os.listdir(path):
    if file.endswith('.csv'):
        shutil.copy(os.path.join(path, file), '.')
        print(f"Loaded file into workspace: {file}")

Running on device: cuda


100%|██████████| 41.0M/41.0M [00:00<00:00, 138MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/emineyetm/fake-news-detection-datasets/versions/1


## Step 1: Data Preprocessing & Cleaning

We load the raw datasets (`True.csv` and `Fake.csv`), assign binary labels (0 for True, 1 for Fake), and merge them.
To address source-cue bias, we define a custom regex-based cleaning function that strips out common publishing agency patterns and datelines (such as "WASHINGTON (Reuters) -") from the beginning of the news texts.

In [9]:
# Load datasets
df_fake = pd.read_csv('Fake.csv')
df_true = pd.read_csv('True.csv')

# Assign binary labels
df_fake['label'] = 1
df_true['label'] = 0

# Combine and shuffle
df = pd.concat([df_fake, df_true], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Combine title and text fields for comprehensive context
df['content'] = df['title'].fillna('') + " " + df['text'].fillna('')

# Function to remove source-cue bias (agency prefixes like 'REUTERS', 'WASHINGTON —')
def remove_source_cues(text):
    # Regex to strip common agency headers typically found at the start of news articles
    pattern = r'^[A-Z\s\-\—\–]+\s*[\-\—\–]\s*'
    cleaned = re.sub(pattern, '', text)
    return cleaned

# Apply cleaning function to create 'cleaned_content'
df['cleaned_content'] = df['content'].apply(remove_source_cues)

# Quick sanity check
print(f"Total dataset size: {len(df)} samples")

Total dataset size: 44898 samples


## Step 2: Vocabulary & Sequence Tokenization

We tokenize the text corpus by building a custom vocabulary mapping words to integer indices.
All input sequences are padded or truncated to a fixed maximum length (`MAX_LEN = 300`) to ensure uniform tensor shapes for our recurrent architecture.

In [10]:
# Tokenization parameters
MAX_VOCAB_SIZE = 20000
MAX_LEN = 300
BATCH_SIZE = 64
EMBEDDING_DIM = 128
HIDDEN_DIM = 64
EPOCHS = 3
LEARNING_RATE = 0.001

# Build simple whitespace tokenizer and vocabulary
def build_vocab(texts, max_size):
    word_counts = {}
    for text in texts:
        for word in text.lower().split():
            word_counts[word] = word_counts.get(word, 0) + 1

    sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for idx, (word, _) in enumerate(sorted_words[:max_size-2]):
        vocab[word] = idx + 2
    return vocab

vocab = build_vocab(df['content'].tolist(), MAX_VOCAB_SIZE)

def text_to_sequence(text, vocab, max_len):
    tokens = text.lower().split()
    seq = [vocab.get(token, vocab['<UNK>']) for token in tokens]
    if len(seq) < max_len:
        seq = seq + [vocab['<PAD>']] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    return seq

# PyTorch Dataset wrapper
class NewsDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.float)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]

## Step 3: BiLSTM Baseline Model (Raw Data)

We implement a Bidirectional LSTM (BiLSTM) classifier in PyTorch.
First, we train and evaluate the model using the raw text data (including original source cues) to establish our primary benchmark.

In [11]:
# Train/Val/Test split for raw data
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['content'].tolist(), df['label'].tolist(), test_size=0.3, random_state=42, stratify=df['label']
)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

X_train = [text_to_sequence(t, vocab, MAX_LEN) for t in train_texts]
X_test = [text_to_sequence(t, vocab, MAX_LEN) for t in test_texts]

train_loader = DataLoader(NewsDataset(X_train, train_labels), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(NewsDataset(X_test, test_labels), batch_size=BATCH_SIZE, shuffle=False)

# BiLSTM Architecture Definition
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(BiLSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        # Concatenate forward and backward final hidden states
        hidden_cat = torch.cat((hidden[-2], hidden[-1]), dim=1)
        out = self.fc(hidden_cat)
        return self.sigmoid(out)

model = BiLSTMClassifier(len(vocab), EMBEDDING_DIM, HIDDEN_DIM).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Training BiLSTM on Raw Data...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for sequences, labels in train_loader:
        sequences, labels = sequences.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(sequences)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {total_loss/len(train_loader):.4f}")

# Evaluation on Raw Data
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for sequences, labels in test_loader:
        sequences = sequences.to(device)
        outputs = model(sequences)
        preds = (outputs >= 0.5).float().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

acc = accuracy_score(all_labels, all_preds)
prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro')

print("\n--- Raw Data BiLSTM Results ---")
print(f"Accuracy:  {acc:.4f}")
print(f"Macro F1:  {f1:.4f}")

Training BiLSTM on Raw Data...
Epoch [1/3], Loss: 0.0952
Epoch [2/3], Loss: 0.0052
Epoch [3/3], Loss: 0.0027

--- Raw Data BiLSTM Results ---
Accuracy:  0.9996
Macro F1:  0.9996


## Step 4: Source-Cue Control Experiment (Cleaned Data)

To quantify how much the model relies on publishing sources rather than linguistic content, we retrain the exact same BiLSTM architecture using our `cleaned_content` dataset.
Comparing these results against our baseline reveals the true robustness of the model.

In [12]:
# Train/Val/Test split for cleaned data
train_texts_c, temp_texts_c, _, _ = train_test_split(
    df['cleaned_content'].tolist(), df['label'].tolist(), test_size=0.3, random_state=42, stratify=df['label']
)
_, test_texts_c, _, _ = train_test_split(
    temp_texts_c, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

X_train_c = [text_to_sequence(t, vocab, MAX_LEN) for t in train_texts_c]
X_test_c = [text_to_sequence(t, vocab, MAX_LEN) for t in test_texts_c]

train_loader_c = DataLoader(NewsDataset(X_train_c, train_labels), batch_size=BATCH_SIZE, shuffle=True)
test_loader_c = DataLoader(NewsDataset(X_test_c, test_labels), batch_size=BATCH_SIZE, shuffle=False)

model_c = BiLSTMClassifier(len(vocab), EMBEDDING_DIM, HIDDEN_DIM).to(device)
optimizer_c = optim.Adam(model_c.parameters(), lr=LEARNING_RATE)

print("Training BiLSTM on Cleaned Data (Source-Cue Control)...")
for epoch in range(EPOCHS):
    model_c.train()
    total_loss = 0
    for sequences, labels in train_loader_c:
        sequences, labels = sequences.to(device), labels.to(device).unsqueeze(1)
        optimizer_c.zero_grad()
        outputs = model_c(sequences)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_c.step()
        total_loss += loss.item()
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {total_loss/len(train_loader_c):.4f}")

# Evaluation on Cleaned Data
model_c.eval()
all_preds_c, all_labels_c = [], []
with torch.no_grad():
    for sequences, labels in test_loader_c:
        sequences = sequences.to(device)
        outputs = model_c(sequences)
        preds = (outputs >= 0.5).float().cpu().numpy()
        all_preds_c.extend(preds)
        all_labels_c.extend(labels.numpy())

acc_c = accuracy_score(all_labels_c, all_preds_c)
prec_c, rec_c, f1_c, _ = precision_recall_fscore_support(all_labels_c, all_preds_c, average='macro')

print("\n--- Cleaned Data BiLSTM Results ---")
print(f"Accuracy:  {acc_c:.4f}")
print(f"Macro F1:  {f1_c:.4f}")

Training BiLSTM on Cleaned Data (Source-Cue Control)...
Epoch [1/3], Loss: 0.0946
Epoch [2/3], Loss: 0.0065
Epoch [3/3], Loss: 0.0027

--- Cleaned Data BiLSTM Results ---
Accuracy:  0.9993
Macro F1:  0.9993
